In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sqlparse
import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [3]:
import duckdb 

from src import logger
from src.config import CFG
from src.db.election_db import ElectionDB
from src.agent import RAGAgent, SQLAgent, HybridAgent, QueryIntent

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


2026-03-12 02:05:18,761 [WARNING] huggingface_hub._login: Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


2026-03-12 02:05:18,966 [WARNING] huggingface_hub._login: Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


/Users/dric/projects/research/LLMs/chat_app/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [55]:
agent = HybridAgent()

agent()

2026-03-12 03:29:12,766 [INFO] LocalLLMStack: Agent Initialized


In [56]:
agent.client.streaming, agent.client.openai_api_base

(False, 'http://127.0.0.1:8000/v1')

### Inspecting tools

In [57]:
for idx, t in enumerate(agent.tools):
    print(f" {idx+1}) {t.name}: \t{t.description[:30]}...")

 1) describe_table: 	Get column names, types, and d...
 2) execute_read_query: 	Executes a SQL SELECT query an...
 3) get_db_schema: 	Returns the schema (table name...
 4) list_tables: 	Get a list of all available ta...
 5) sample_data: 	Fetch the first 3 rows of a ta...
 6) validate_sql: 	Validating SQL syntax and logi...
 7) search_database: 	Performs hybrid (Full-Text and...


## Check DB search operations

In [ ]:
agent.sql_expert.list_tables.args

In [ ]:
agent.sql_expert.list_tables.invoke({
    "reasoning": "just listing tables"
})

In [ ]:
agent.sql_expert.sample_data.args

In [ ]:
agent.sql_expert.sample_data.invoke({
    "reasoning": "sampling some data to see the values",
    "table_name": "vw_results",
    "num_samples": 10
})

In [ ]:
agent.sql_expert.describe_table.args

In [ ]:
agent.sql_expert.describe_table.invoke({
    "reasoning": "describing the table structure",
    "table_name": "vw_party"
})

### Vector search (VS)

In [ ]:
results = agent.db_client.vector_search(
    reasoning="performing some vector search",
    query="Who was elected in the Agneby-Tiassa region from RHDP?", 
    top_k=5
)

for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

### Full text search (FTS)

In [ ]:
results = agent.db_client.full_text_search(
    query="What was the turnout in tiapoum?", 
    top_k=5
)

for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

### Hybrid search

In [ ]:
results = agent.db_client.hybrid_search(
    query="Who was elected in Tiapoum region?", 
    top_k=5
)

for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

## Check Agent-specific operations

### Testing Security Sandbox

In [58]:
sql = "SELECT * FROM REGION"
parsed = sqlparse.parse(sql)

clean_sql = parsed[0]

clean_sql, clean_sql.value

(<Statement 'SELECT...' at 0x12ED69520>, 'SELECT * FROM REGION')

In [59]:
agent.sql_expert.validate_sql.args

{'sql': {'title': 'Sql', 'type': 'string'},
 'metrics': {'additionalProperties': True,
  'title': 'Metrics',
  'type': 'object'},
 'forbidden': {'items': {}, 'title': 'Forbidden', 'type': 'array'},
 'reasoning': {'title': 'Reasoning', 'type': 'string'}}

In [60]:
try:
    agent.forbidden = ["SELECT"] # Attempt to change the list
    agent._forbidden = [] # Attempt to change the list

except AttributeError as e:
    print(f"✅ Success: Modification blocked - {e}")

# Manual validation test
for kw in ["DROP", "ALTER", "UPDATE"]:
    malicious = f"SELECT * FROM results; {kw} TABLE turnout;"
    is_safe, out_message = agent.sql_expert.validate_sql.invoke({
        "reasoning": "validating SQL query",
        "sql": malicious,
        "metrics": agent.sql_expert.metrics,
        "forbidden": agent.sql_expert.forbidden
    })
    print(f"Is {kw} query safe? {is_safe} (Expected: False)")
    print(f"Message: {out_message}")
    print()

for kw in ["DROP", "ALTER", "UPDATE"]:
    malicious = f"{kw} TABLE result;"
    is_safe, out_message = agent.sql_expert.validate_sql.invoke({
        "reasoning": "validating SQL query",
        "sql": malicious,
        "metrics": agent.sql_expert.metrics,
        "forbidden": agent.sql_expert.forbidden
    })
    print(f"Is {kw} query safe?\t {is_safe} (Expected: False)")
    print(f"Message: {out_message}")
    print()

✅ Success: Modification blocked - The 'forbidden' attribute is sealed and cannot be modified.
2026-03-12 03:29:18,310 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is DROP query safe? False (Expected: False)
Message: Security Violation: Multiple queries detected.

2026-03-12 03:29:18,312 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is ALTER query safe? False (Expected: False)
Message: Security Violation: Multiple queries detected.

2026-03-12 03:29:18,316 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is UPDATE query safe? False (Expected: False)
Message: Security Violation: Multiple queries detected.

2026-03-12 03:29:18,320 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is DROP query safe?	 False (Expected: False)
Message: Security Violation: Forbidden command type 'DROP'.

2026-03-12 03:29:18,322 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is ALTER 

In [61]:
is_safe, out_message = agent.sql_expert.validate_sql.invoke({
    "reasoning": "validating SQL query",
    "sql": clean_sql.value,
    "metrics": agent.sql_expert.metrics,
    "forbidden": agent.sql_expert.forbidden
})
print(f"Is query safe? {is_safe} (Expected: False)")
print(f"Message: {out_message}")
print()

2026-03-12 03:29:19,347 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is query safe? False (Expected: False)
Message: Security Violation: Unauthorized table access detected.
SQL: SELECT * FROM REGION



In [62]:
is_valid, msg = agent.sql_expert.validate_sql.invoke({
    "sql": "SELECT PARTY_NAME, SUM(SCORES) AS total_votes FROM result GROUP BY PARTY_ID ORDER BY total_votes DESC LIMIT 20",
    "reasoning": "validating SQL query",
    "metrics": agent.sql_expert.metrics,
    "forbidden": agent.sql_expert.forbidden
})

print(f"Is query safe? {is_safe} (Expected: False)")
print(f"Message: {out_message}")

2026-03-12 03:29:20,792 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is query safe? False (Expected: False)
Message: Security Violation: Unauthorized table access detected.
SQL: SELECT * FROM REGION


In [48]:
agent.sql_expert.execute_read_query.args

{'sql_query': {'title': 'Sql Query', 'type': 'string'},
 'reasoning': {'title': 'Reasoning', 'type': 'string'}}

In [ ]:
results, out_msg = agent.sql_expert.execute_read_query.invoke({
    "sql_query": "SELECT PARTY_NAME, SUM(SCORES) AS total_votes FROM result GROUP BY PARTY_ID ORDER BY total_votes DESC LIMIT 20",
    "reasoning": "checking query"
})

print(f"{results=}")
print(f"{out_msg=}")

### Testing intent routing

In [ ]:
test_queries = [
    ("Who got the most votes in Abidjan?", "RANKING"),
    ("What is the total turnout?", "AGGREGATION"),
    ("How many colors are there in the rainbow?", "INVALID"),
    ("Tell me about the history of the 2020 election.", "GENERAL"),
    ("What is the weather in Paris?", "INVALID")
]

for query, expected in test_queries:
    intent = agent._get_intent(query)
    print(f"Query: {query} \n> Detected: {intent} (Expected: {expected})\n")
    print()


### SQLAgent 

In [7]:
# import sys

# # Test a raw stream
# for chunk in agent.client.stream("Write a 2-sentence story about PhD."):
#     sys.stdout.write(chunk.content)
#     sys.stdout.flush()


In [14]:
%%time 

query = "What is the total votes by party ?"
intent=QueryIntent.AGGREGATION

if CFG.IS_STREAM:
    print(f"--- Starting Stream for: {query} ---\n")
    
    for chunk in agent.sql_expert.generate_sql(query, intent):
        if chunk["type"] == "status":
            print(f"\n\n[STATUS]: {chunk['content']}", end="", flush=True)
        
        elif chunk["type"] == "token":
            print(chunk["content"], end="", flush=True)
            
        elif chunk["type"] == "action":
            # Visual separator for tool calls
            print(f"\n\n🛠️  [SYSTEM]: {chunk['content']}", end="", flush=True)
            
        elif chunk["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{chunk['content']}", end="", flush=True)
            final_sql = chunk['content']
            print(f"\n\n🚀 [SUCCESS]: SQL Captured.")
    
        elif chunk["type"] == "error":
            print(f"\n❌ [ERROR]: {chunk['content']}", end="", flush=True)

else:
    for out in agent.sql_expert.generate_sql(query, intent):
        print(out)
        response = out

2026-03-12 02:11:16,231 [INFO] LocalLLMStack: Attempting SQL generation...(Max iterations=18)
{'type': 'status', 'content': 'Attempting SQL generation...(Max iterations=18)'}
2026-03-12 02:11:16,232 [INFO] LocalLLMStack: 
Starting iteration 1/18
{'type': 'status', 'content': '\nStarting iteration 1/18'}
2026-03-12 02:12:09,491 [INFO] LocalLLMStack: Extracting tool calls
2026-03-12 02:12:09,492 [INFO] LocalLLMStack: Executing tool: list_tables with args {'reasoning': 'To identify the relevant tables that might contain information about votes and parties, I first need to list all available tables in the database. This will help me determine which tables are likely to hold data on votes and party affiliations.'}
{'type': 'status', 'content': "Executing tool: list_tables with args {'reasoning': 'To identify the relevant tables that might contain information about votes and parties, I first need to list all available tables in the database. This will help me determine which tables are likel

In [15]:
response

{'type': 'final_sql',
 'content': 'SELECT SUM(SCORES) AS TOTAL_VOTES, PARTY_NAME FROM vw_results GROUP BY PARTY_NAME ORDER BY TOTAL_VOTES DESC LIMIT 50',
 'sanitized_query': 'SELECT SUM(SCORES) AS TOTAL_VOTES, PARTY_NAME FROM vw_results GROUP BY PARTY_NAME ORDER BY TOTAL_VOTES DESC LIMIT 50'}

In [16]:
pprint(response["sanitized_query"])

('SELECT SUM(SCORES) AS TOTAL_VOTES, PARTY_NAME FROM vw_results GROUP BY '
 'PARTY_NAME ORDER BY TOTAL_VOTES DESC LIMIT 50')


In [17]:
results, out_msg = agent.sql_expert.execute_read_query.invoke({
    "sql_query": response["sanitized_query"],
    "reasoning": "checking query"
})

print(f"{results.shape=}")
print(f"{out_msg=}")

2026-03-12 02:15:00,046 [INFO] LocalLLMStack: LLM Reasoning (execute_read_query): checking query
results.shape=(37, 2)
out_msg='OK'


In [18]:
results

,TOTAL_VOTES,PARTY_NAME
0,3293852.0,RHDP
1,2409761.0,PDCI-RDA
2,684183.0,INDEPENDANT
3,14143.0,FPI
4,11883.0,ADCI
5,4771.0,CODE
6,3844.0,MGC
7,3387.0,EDS
8,2571.0,UNPR
9,1945.0,AIDE


In [19]:
intent = agent._get_intent(user_prompt=query)
print(intent)

out = agent._interpret_results(
    user_prompt=query,
    df=results,
    intent=intent
)

pprint(out) 

2026-03-12 02:15:00,149 [INFO] LocalLLMStack: Formatting input to LLM
QueryIntent.AGGREGATION
2026-03-12 02:15:04,579 [INFO] LocalLLMStack: Interpreting results...
2026-03-12 02:15:04,582 [INFO] LocalLLMStack: Prompting LLM
('The total votes by party show a clear dominance by the RHDP and PDCI-RDA, '
 'which together account for over 5.7 million votes—more than 90% of the '
 'total. The RHDP alone received approximately 3.29 million votes, making it '
 'the largest party. The PDCI-RDA followed with about 2.41 million votes. The '
 'remaining parties collectively represent a small fraction of the total, with '
 'most having fewer than 1,000 votes. This indicates a highly concentrated '
 'electoral landscape, where a few major parties capture the vast majority of '
 'the vote.')


In [26]:
query = "Which region has the most voters?"
intent = QueryIntent.RANKING  # Manually set for the test
final_sql_query = None

if CFG.IS_STREAM:
    print(f"--- Starting Stream for: {query} ---\n")
    
    for chunk in agent.sql_expert.generate_sql(query, intent):
        
        if chunk["type"] == "token":
            print(chunk["content"], end="", flush=True)
            
        elif chunk["type"] == "action":
            print(f"\n\n🛠️  [ACTION]: {chunk['content']}")
            
        elif chunk["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{chunk['content']}")
            final_sql = chunk['content']
    
    print("\n--- Stream Finished ---")
else:
    for out in agent.sql_expert.generate_sql(query, intent):
        print(out)
        response = out

2026-03-12 02:29:52,272 [INFO] LocalLLMStack: Attempting SQL generation...(Max iterations=18)
{'type': 'status', 'content': 'Attempting SQL generation...(Max iterations=18)'}
2026-03-12 02:29:52,272 [INFO] LocalLLMStack: 
Starting iteration 1/18
{'type': 'status', 'content': '\nStarting iteration 1/18'}
2026-03-12 02:30:44,088 [INFO] LocalLLMStack: Extracting tool calls
2026-03-12 02:30:44,091 [INFO] LocalLLMStack: Executing tool: list_tables with args {'reasoning': 'To identify the relevant tables that might contain information about regions and voter turnout, I first need to list all available tables in the database. This will help me determine which table(s) are likely to hold data on regions and voter numbers.'}
{'type': 'status', 'content': "Executing tool: list_tables with args {'reasoning': 'To identify the relevant tables that might contain information about regions and voter turnout, I first need to list all available tables in the database. This will help me determine which t

In [27]:
response

{'type': 'final_sql',
 'content': 'SELECT \n    REGION_NAME,\n    NUM_VOTERS\nFROM \n    vw_turnout\nORDER BY \n    NUM_VOTERS DESC\nLIMIT 50;',
 'sanitized_query': 'SELECT\n    REGION_NAME,\n    NUM_VOTERS\nFROM\n    vw_turnout\nORDER BY\n    NUM_VOTERS DESC\nLIMIT 50'}

In [28]:
response["sanitized_query"]

'SELECT\n    REGION_NAME,\n    NUM_VOTERS\nFROM\n    vw_turnout\nORDER BY\n    NUM_VOTERS DESC\nLIMIT 50'

In [29]:
results, out_msg = agent.sql_expert.execute_read_query.invoke({
    "sql_query": response["sanitized_query"],
    "reasoning": "checking query"
})

print(f"{results.shape=}")
print(f"{out_msg=}")

2026-03-12 02:32:36,822 [INFO] LocalLLMStack: LLM Reasoning (execute_read_query): checking query
results.shape=(50, 2)
out_msg='OK'


In [30]:
results

,REGION_NAME,NUM_VOTERS
0,PORO,142250
1,GBEKE,114172
2,DISTRICTAUTONOMED'ABIDJAN,107134
3,GUEMON,73989
4,DISTRICTAUTONOMED'ABIDJAN,73989
5,TCHOLOGO,40576
6,BAGOUE,36846
7,DISTRICTAUTONOMEDEYAMOUSSOUKRO,36304
8,DISTRICTAUTONOMED'ABIDJAN,33793
9,DISTRICTAUTONOMED'ABIDJAN,33090


In [31]:
intent = agent._get_intent(user_prompt=query)
print(intent)

out = agent._interpret_results(
    user_prompt=query,
    df=results,
    intent=intent
)

pprint(out) 

2026-03-12 02:32:36,975 [INFO] LocalLLMStack: Formatting input to LLM
QueryIntent.RANKING
2026-03-12 02:32:40,103 [INFO] LocalLLMStack: Interpreting results...
2026-03-12 02:32:40,106 [INFO] LocalLLMStack: Prompting LLM
('The region with the most voters is **PORO**, with a total of **142,250** '
 'voters. This significantly outpaces the second-highest region, **GBEKE** '
 '(114,172 voters), highlighting a substantial gap in voter numbers. \n'
 '\n'
 "Notably, **DISTRICTAUTONOMED'ABIDJAN** appears multiple times across the "
 'dataset, with several entries in the 30,000–40,000 voter range, indicating '
 'it is a major electoral area but still behind PORO. Other regions like '
 'GUEMON and TCHOLOGO show repeated entries with lower voter counts, '
 'suggesting possible data duplication or reporting inconsistencies.\n'
 '\n'
 'The data reveals a clear concentration of voters in PORO and GBEKE, with '
 'significant disparities among the rest of the regions—many of which have '
 'fewer than 

In [ ]:
query = "What is the turnout in the region with the most voters?"
intent = QueryIntent.AGGREGATION 

if CFG.IS_STREAM:

    print("--- [STARTING STREAM] ---")
    for update in agent.sql_expert.generate_sql(query, intent):
        
        if update["type"] == "token":
            print(update["content"], end="", flush=True)
            
        elif update["type"] == "action":
            print(f"\n\n🛠️  [TOOL CALL]: {update['content']}")
            
        elif update["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{update['content']}")
    
    print("\n--- [STREAM FINISHED] ---")
else:
    response = agent.sql_expert.generate_sql(query, intent)

### Full End-to-End Analytical Run

In [15]:
vague_query = "Who the candidates who won the elections in Abidjan?" 
# Logic: Abidjan is a wide region so agent might need clarification

agent._get_intent(user_prompt=vague_query)

2026-03-11 13:06:46,082 [INFO] LocalLLMStack: Formatting input to LLM


<QueryIntent.RANKING: 'ranking'>

In [16]:
agent.rag_expert.search_database.args

{'query': {'title': 'Query', 'type': 'string'}}

In [17]:
results = agent.rag_expert.search_database.invoke({    
    "query": vague_query
    })

2026-03-11 13:06:52,091 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: mps
2026-03-11 13:06:52,091 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: google/embeddinggemma-300m


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

2026-03-11 13:07:03,042 [INFO] sentence_transformers.SentenceTransformer: 14 prompts are loaded, with the keys: ['query', 'document', 'BitextMining', 'Clustering', 'Classification', 'InstructionRetrieval', 'MultilabelClassification', 'PairClassification', 'Reranking', 'Retrieval', 'Retrieval-query', 'Retrieval-document', 'STS', 'Summarization']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

chunk_id 173: 	[0.02] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, ABOBO RENAISSANCE (INDEPENDANT) received 916 votes (0.87%) but has lost.
chunk_id 174: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, AGIR AUJOURDHUI POUR BATIR DEMAIN (AIDE) received 1374 votes (1.31%) but has lost.
chunk_id 1125: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, UNE CÔTE DIVOIRE EN PAIX, PROSPERE ET SOLIDAIRE (RHDP) received 92947 votes (88.45%) but has lost.
chunk_id 178: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, TOUS ENSEMBLE POUR LA CÔTE D'IVOIRE (PDCI-RDA) received 7359 votes (7.0%) but has lost.
chunk_id 1722: 	[0.01] The voter turnout in constituency 048 (DISTRICTAUTONOMED'ABIDJAN) was 21.22%. There were 107529 registered voters and 22226 expressed votes including 235 blank votes (1.06%).
chunk_id 175: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTR

### get_answer()

In [19]:
print(f"--- Testing Disambiguation for: {vague_query} ---")
try:
    for response in agent.get_answer(vague_query):

        if response.get("type") == "clarification":
            print("Success: Agent identified ambiguity.")
            print("Agent Question:", response["content"])
            print("Options found in DB:", response["options"])

except Exception as e:
    logger.error(f"Failure: Agent failed to process query. {e}", exc_info=True)


--- Testing Disambiguation for: Who the candidates who won the elections in Abidjan? ---
2026-03-11 13:07:17,510 [INFO] LocalLLMStack: Formatting input to LLM
2026-03-11 13:07:21,619 [INFO] LocalLLMStack: Identifying decision route using LLM routing...
2026-03-11 13:07:25,936 [INFO] LocalLLMStack: ✅ Identified route: SQL
2026-03-11 13:07:25,937 [INFO] LocalLLMStack: Generating SQL query
2026-03-11 13:07:25,937 [INFO] LocalLLMStack: Attempting restricted SQL generation...(Max iterations=18)
2026-03-11 13:07:25,938 [ERROR] LocalLLMStack: Query generation error: 'StructuredTool' object is not callable


In [13]:
response

{'type': 'status',
 'content': "Could not retrieve response. 'str' object has no attribute 'hybrid_search'"}

#### Test various queries

In [ ]:
queries = [
    "How many seats did RHDP win?",
    "Who won the elections in tiapum",
    "Top 10 candidates by score in region Nawa.",
    "Participation rate by region",
    "Histogram of winners by party.",
    "distribution of winners per party",
    "Which party did win the most seats?",
    "Show me the distribution of voters per party and per region"
]

for query in queries:
    print(f'>> Query: {query}')
    out = agent.get_answer(user_prompt=query)
    print(f'>> Response: {out["content"]}\n')
    break
